# Arez — Devil's Advocate (@Arez_provocateur_bot)
**Model:** Mistral Small 3.1 24B · **Hardware:** A100 40GB

| Bước | Cell | Làm gì |
|---|---|---|
| 1 | **Cell 1** | Cài thư viện → **Restart Runtime** |
| 2 | **Cell 2** | Load model (~3 phút) |
| 3 | **Cell 4** | Chạy Telegram bot — nhắn @Arez_provocateur_bot là được |
| Alt | **Cell 3** | Chat trực tiếp trong Colab (không cần Telegram) |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN
# ⚠️ SAU KHI XONG: Runtime > Restart runtime
# ═══════════════════════════════════════════════════

import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-telegram-bot==20.7'])

print('✅ Cài xong.')
print('⚠️  Bây giờ: Runtime > Restart runtime → chạy Cell 2')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — LOAD MODEL
# Chạy 1 lần sau restart. ~3 phút.
# ═══════════════════════════════════════════════════

from vllm import LLM, SamplingParams

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

print(f'Đang load {MODEL_ID}...')
llm = LLM(
    model=MODEL_ID,
    dtype='bfloat16',
    max_model_len=8192,
    gpu_memory_utilization=0.88,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
    repetition_penalty=1.1,
)

SYSTEM_PROMPT = (
    "Bạn là Arez — AI đóng vai Devil's Advocate.\n\n"
    "Nhiệm vụ:\n"
    "- Phản biện mọi luận điểm, quyết định, kế hoạch\n"
    "- Tìm điểm yếu, rủi ro ẩn, giả định sai\n"
    "- Đặt câu hỏi sắc bén, trực tiếp\n"
    "- KHÔNG đồng ý dễ dàng\n"
    "- Trả lời tiếng Việt, ngắn gọn, đi thẳng vào vấn đề\n\n"
    "Mục tiêu: làm lập luận của người dùng vững hơn."
)

# Lưu lịch sử per user_id
histories = {}

def ask(user_text, user_id='default'):
    if user_id not in histories:
        histories[user_id] = []
    histories[user_id].append({'role': 'user', 'content': user_text})

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + histories[user_id]
    prompt = llm.get_tokenizer().apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    outputs = llm.generate([prompt], sampling_params)
    response = outputs[0].outputs[0].text.strip()
    histories[user_id].append({'role': 'assistant', 'content': response})
    return response

# Test nhanh
print('\n── Test ──')
test_q = 'Tôi muốn đầu tư 10 tỷ vào bất động sản Thủ Đức vì giá đang tăng.'
print(f'Anh: {test_q}')
print(f'Arez: {ask(test_q)}')
histories.clear()
print('\n✅ Model sẵn sàng. Chạy Cell 4 để bật Telegram bot.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — CHAT TRONG COLAB (không cần Telegram)
# Sửa USER_INPUT → Run mỗi lần hỏi
# ═══════════════════════════════════════════════════

USER_INPUT = """Nhập câu hỏi hoặc luận điểm ở đây."""

print(f'Anh: {USER_INPUT}')
print()
print('Arez:', ask(USER_INPUT))

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — TELEGRAM BOT
# Chạy sau Cell 2. Để cell chạy — bot hoạt động liên tục.
# Nhắn @Arez_provocateur_bot trên Telegram là được.
# ═══════════════════════════════════════════════════

import asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, CommandHandler, filters, ContextTypes

BOT_TOKEN = '8747730907:AAFTAb7I6r7T36Da6bpX1eVT9mEfg-rOkrM'

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = str(update.effective_user.id)
    if user_id in histories:
        histories[user_id].clear()
    await update.message.reply_text(
        '🔴 *Arez đây.*\n\n'
        'Tôi không đồng ý với bạn cho đến khi bạn chứng minh được lập luận của mình.\n\n'
        'Nói đi.',
        parse_mode='Markdown'
    )

async def reset(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = str(update.effective_user.id)
    if user_id in histories:
        histories[user_id].clear()
    await update.message.reply_text('🔄 Lịch sử đã reset. Bắt đầu lại.')

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_text = update.message.text
    user_id = str(update.effective_user.id)
    user_name = update.effective_user.first_name or 'Unknown'

    if not user_text:
        return

    # Typing indicator
    await context.bot.send_chat_action(
        chat_id=update.effective_chat.id,
        action='typing'
    )

    print(f'[{user_name}]: {user_text[:80]}')

    # Generate response
    response = ask(user_text, user_id)

    print(f'[Arez]: {response[:80]}...')

    # Split nếu response > 4096 ký tự (Telegram limit)
    if len(response) <= 4096:
        await update.message.reply_text(response)
    else:
        for i in range(0, len(response), 4096):
            await update.message.reply_text(response[i:i+4096])

# Build app
app = ApplicationBuilder().token(BOT_TOKEN).build()
app.add_handler(CommandHandler('start', start))
app.add_handler(CommandHandler('reset', reset))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

print('✅ @Arez_provocateur_bot đang chạy...')
print('   Nhắn bot trên Telegram để bắt đầu.')
print('   /start — bắt đầu | /reset — reset lịch sử')
print('   Ctrl+C hoặc interrupt kernel để dừng.')

await app.run_polling()